# Leave-one-bioreactor-out PINN Benchmark

## 1. Configure Training


In [1]:
import importlib
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
import src.experiments.leave_one_out as loo_module
import src.utils.dataloader as dataloader_module
import src.utils.training as training_module

importlib.reload(dataloader_module)
importlib.reload(training_module)
importlib.reload(loo_module)

from src.utils.training import TrainingConfig
from src.models.pinn import PseudomonasBIOSODE

run_loo = loo_module.run_leave_one_bioreactor_out
summarize_leave_one_out = loo_module.summarize_leave_one_out

csv_path = "data/processed/ambr_preprocessed.csv"
out_dir = "results"

ids = (
    "AMBR1_14", "AMBR1_15", "AMBR1_16", "AMBR1_17", "AMBR1_18",
    "AMBR1_21", "AMBR1_22", "AMBR1_23", "AMBR1_24",
    "AMBR2_5", "AMBR2_6", "AMBR2_7", "AMBR2_8", "AMBR2_9", 
    "AMBR2_10",
    # "AMBR2_12", 
    "AMBR2_13", "AMBR2_14", "AMBR2_15", "AMBR2_16", "AMBR2_19", 
    "AMBR2_20", "AMBR2_21",
)
 
seeds = (42,)
epochs = 10000
fold_indices = None
cfg = TrainingConfig(
    processed_csv=csv_path,
    results_dir=out_dir,
    experiment_ids=ids,
    experiment_name="LOO",
    seed=seeds[0],
    n_neurons=20,
    n_hidden_layers=7,
    num_epochs=epochs,
    learning_rate=1e-4,
    data_loss_weight=1.0,
    residual_loss_weight=1.0,
    auxiliary_loss_weight=1.0,
    regularization_loss_weight=1.0,
    use_auxiliary_loss=True,
    use_regularization_loss=False,
    use_softadapt=True,
    track_epoch_r2=False,
    use_early_stopping=False,
    early_stopping_patience=1000,
    early_stopping_min_delta=1e-4,
    restore_best_weights=True,
    validation_strategy="random",
    validation_seed=seeds[0],
    obs_fit_weights=(1.0, 1.0, 0.1, 0.1),  # glucose, biomass, O2, pH
    aux_fit_weights=(1.0,) * len(PseudomonasBIOSODE.state_names),
    res_eq_weights=(1e-15,) * len(PseudomonasBIOSODE.state_names),
    reg_eq_weights=(1e-6,) * len(PseudomonasBIOSODE.state_names),
)


## 2. Run Training


In [ ]:
loo = run_loo(
    cfg,
    seeds=seeds,
    fold_indices=fold_indices,
    keep_results=True,
)

results = loo["results"]

## 3. Build Summary

In [3]:
results_dir = PROJECT_ROOT / out_dir / cfg.experiment_name
summary = summarize_leave_one_out(results_dir)
summary_path = summary["table"]
summary_df = pd.read_csv(summary_path)
display(summary_df)

,Benchmark,Observable,R2 mean,R2 median,R2 std,MAE mean,MAE median,MAE std,NRMSE mean,NRMSE median,NRMSE std
0,Leave-One-Bioreactor-Out,Biomass / OD,0.736762,0.844550,0.275328,0.779567,0.600547,0.566508,0.085606,0.066413,0.058012
1,Leave-One-Bioreactor-Out,Glucose,0.892904,0.962798,0.160781,0.605895,0.452750,0.514225,0.045273,0.033701,0.035487
2,Leave-One-Bioreactor-Out,DO,0.421699,0.432577,0.357308,8.876669,7.878124,6.458580,0.080714,0.067405,0.057481
3,Leave-One-Bioreactor-Out,pH,0.088082,0.000000,0.233070,0.112802,0.074854,0.110894,0.058553,0.039063,0.053996
